<a href="https://colab.research.google.com/github/0se0/hw2/blob/develop/ML_h2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bank Customer Churn Prediction — Advanced Ensemble

Kaggle 스타일 이진 분류(고객 이탈 예측) 파이프라인.
RF / ExtraTrees / GradientBoosting / LogisticRegression + (설치되어 있으면) LightGBM / XGBoost / CatBoost를
5-fold Out-of-Fold(OOF) 방식으로 학습하고, 가중 블렌드와 스태킹 중 더 나은 쪽을 최종 앙상블로 채택합니다.

로컬 환경과 Google Colab 양쪽에서 동작합니다. 로컬에서 실행할 때는 `train.csv`, `test.csv`를
이 노트북과 같은 폴더에 두거나 `CHURN_DATA_DIR` 환경 변수로 경로를 지정하세요.

In [ ]:
# =============================================================================
# 🔧 1. 환경 설정 및 라이브러리 import
# =============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트가 있으면 사용 (없는 환경에서는 그냥 기본 폰트로 넘어감 -- 글자가
# 깨지는 것 말고는 기능에 영향 없음)
for _font in ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR']:
    if _font in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _font
        break
plt.rcParams['axes.unicode_minus'] = False
from scipy.optimize import differential_evolution

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib

# 부스팅 라이브러리는 설치되어 있는 것만 사용 (없으면 건너뜀)
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("⚠️ lightgbm이 설치되어 있지 않아 건너뜁니다. (pip install lightgbm)")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠️ xgboost가 설치되어 있지 않아 건너뜁니다. (pip install xgboost)")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("⚠️ catboost가 설치되어 있지 않아 건너뜁니다. (pip install catboost)")

RANDOM_STATE = 42
N_FOLDS = 5
DATA_DIR = os.environ.get('CHURN_DATA_DIR', '.')

In [ ]:
# =============================================================================
# 📁 2. 데이터 로드 (로컬 우선, 없으면 Colab 업로드로 폴백)
# =============================================================================

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    if _in_colab():
        from google.colab import files
        print("train.csv 파일을 업로드하세요:")
        files.upload()
        print("test.csv 파일을 업로드하세요:")
        files.upload()
        train_path, test_path = 'train.csv', 'test.csv'
    else:
        raise FileNotFoundError(
            f"train.csv / test.csv를 찾을 수 없습니다. "
            f"'{DATA_DIR}' 폴더에 두거나 CHURN_DATA_DIR 환경 변수로 경로를 지정하세요."
        )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("✅ 데이터 로드 완료!")
print(f"   Train: {train.shape[0]:,} rows, {train.shape[1]} columns")
print(f"   Test: {test.shape[0]:,} rows, {test.shape[1]} columns")
print(f"   이탈률: {train['Exited'].mean():.2%}")

In [ ]:
# =============================================================================
# 📊 3. 탐색적 데이터 분석
# =============================================================================
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
train['Exited'].value_counts().plot(kind='bar')
plt.title('타겟 분포')
plt.xticks([0, 1], ['Stay', 'Churn'], rotation=0)

plt.subplot(1, 3, 2)
train.groupby('Geography')['Exited'].mean().plot(kind='bar')
plt.title('지역별 이탈률')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
train.groupby('Gender')['Exited'].mean().plot(kind='bar')
plt.title('성별 이탈률')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 🔧 4. 피처 엔지니어링
# =============================================================================

def advanced_feature_engineering(df, balance_q80):
    '''고급 피처 엔지니어링.

    balance_q80은 train 데이터에서 계산해 train/test에 동일하게 적용한다.
    (기존 코드는 이 분위수를 train/test 각각에서 따로 계산해 서로 다른
     기준이 적용되는 버그가 있었다 -- test 분포가 train과 다르면 Is_High_Value가
     엉뚱한 기준으로 채워진다.)
    '''
    df = df.copy()
    df = df.drop(['CustomerId', 'Surname'], axis=1, errors='ignore')

    geo_mapping = {'France': 0, 'Germany': 1, 'Spain': 2}
    gender_mapping = {'Female': 0, 'Male': 1}
    df['Geography'] = df['Geography'].map(geo_mapping)
    df['Gender'] = df['Gender'].map(gender_mapping)

    # === 비율 피처 ===
    df['Balance_per_Product'] = df['Balance'] / (df['NumOfProducts'] + 1)
    df['Balance_per_Salary'] = df['Balance'] / (df['EstimatedSalary'] + 1)
    df['CreditScore_per_Age'] = df['CreditScore'] / df['Age']
    df['Salary_per_Age'] = df['EstimatedSalary'] / df['Age']
    df['Tenure_per_Age'] = df['Tenure'] / df['Age']

    # === 상호작용 피처 ===
    df['Age_CreditScore_Interaction'] = df['Age'] * df['CreditScore'] / 1000
    df['Balance_CreditScore_Interaction'] = df['Balance'] * df['CreditScore'] / 1000000
    df['Active_Card_Interaction'] = df['HasCrCard'] * df['IsActiveMember']

    # === 이진 분류 피처 ===
    df['Is_High_Value'] = (df['Balance'] > balance_q80).astype(int)
    df['Is_Zero_Balance'] = (df['Balance'] == 0).astype(int)
    df['Is_Young'] = (df['Age'] < 35).astype(int)
    df['Is_Senior'] = (df['Age'] >= 55).astype(int)
    df['Is_High_Credit'] = (df['CreditScore'] > 700).astype(int)
    df['Is_Multi_Product'] = (df['NumOfProducts'] > 2).astype(int)
    df['Is_Germany'] = (df['Geography'] == 1).astype(int)

    # === 위험 점수 ===
    df['Churn_Risk_Score'] = (
        df['Is_Germany'] * 3 +
        df['Gender'] * 2 +
        df['Is_Senior'] * 2 +
        df['Is_Multi_Product'] * 3 +
        (1 - df['IsActiveMember']) * 2 +
        df['Is_Zero_Balance'] * 1
    )

    # === 복합 지표 ===
    df['Customer_Value_Score'] = (
        df['Balance'] / 100000 +
        df['CreditScore'] / 1000 +
        df['EstimatedSalary'] / 100000 +
        df['NumOfProducts'] * 0.5
    )

    for col in df.columns:
        if col not in ['Exited', 'id']:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df

balance_q80 = train['Balance'].quantile(0.8)
train_processed = advanced_feature_engineering(train, balance_q80)
test_processed = advanced_feature_engineering(test, balance_q80)

feature_cols = [c for c in train_processed.columns if c not in ['Exited', 'id']]
X = train_processed[feature_cols]
y = train_processed['Exited']
X_test = test_processed[feature_cols]

original_features = len([c for c in train.columns if c not in ['Exited', 'id']])
print(f"✅ 피처 엔지니어링 완료!")
print(f"   {original_features}개 → {len(feature_cols)}개 (+{len(feature_cols) - original_features}개 추가)")
print(f"   피처: {feature_cols}")

In [ ]:
# =============================================================================
# 🤖 5. 모델 정의
# =============================================================================

def create_models():
    models = {
        'rf': RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_split=20,
            min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1,
        ),
        'et': ExtraTreesClassifier(
            n_estimators=200, max_depth=12, min_samples_split=20,
            min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1,
        ),
        'gb': GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE,
        ),
        'lr': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=0.1),
    }
    if HAS_LGBM:
        models['lgbm'] = LGBMClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=6, num_leaves=31,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
            n_jobs=-1, verbose=-1,
        )
    if HAS_XGB:
        models['xgb'] = XGBClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    if HAS_CATBOOST:
        models['catboost'] = CatBoostClassifier(
            iterations=400, learning_rate=0.03, depth=6,
            random_state=RANDOM_STATE, verbose=False,
        )
    return models

models = create_models()
model_names = list(models.keys())
print(f"✅ 사용할 모델: {model_names}")

In [ ]:
# =============================================================================
# 🎯 6. Out-of-Fold(OOF) 교차검증 예측 생성
#
#   기존 코드는 train/val을 한 번만 나눠 그 val 기준으로 앙상블 가중치를 정하고,
#   테스트 예측도 그 한 번 학습한 모델로만 만들었다. 이 방식은
#     (a) 가중치가 특정 분할에 과적합될 위험이 있고
#     (b) 테스트 예측의 분산이 크다(모델 1개 = 시드/분할에 따라 흔들림).
#   대신 5-fold 전체에 대해 OOF 예측을 만들어 가중치를 훨씬 안정적으로 추정하고,
#   테스트 예측은 5개 fold 모델의 평균(bagging)으로 분산을 낮춘다.
# =============================================================================

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(cv.split(X, y))

oof_preds = {name: np.zeros(len(X)) for name in model_names}
test_preds = {name: np.zeros(len(X_test)) for name in model_names}
fold_aucs = {name: [] for name in model_names}

for name, base_model in models.items():
    print(f"   {name.upper()} — {N_FOLDS}-fold 학습 중...")
    for fold, (tr_idx, val_idx) in enumerate(fold_indices):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model_fold = base_model.__class__(**base_model.get_params())

        if name == 'lr':
            scaler = StandardScaler().fit(X_tr)
            model_fold.fit(scaler.transform(X_tr), y_tr)
            val_pred = model_fold.predict_proba(scaler.transform(X_val))[:, 1]
            test_pred = model_fold.predict_proba(scaler.transform(X_test))[:, 1]
        else:
            model_fold.fit(X_tr, y_tr)
            val_pred = model_fold.predict_proba(X_val)[:, 1]
            test_pred = model_fold.predict_proba(X_test)[:, 1]

        oof_preds[name][val_idx] = val_pred
        test_preds[name] += test_pred / N_FOLDS
        fold_aucs[name].append(roc_auc_score(y_val, val_pred))

    fold_aucs[name] = np.array(fold_aucs[name])
    oof_auc = roc_auc_score(y, oof_preds[name])
    print(f"     OOF AUC: {oof_auc:.4f}  (fold별: {[f'{a:.4f}' for a in fold_aucs[name]]})")

In [ ]:
# =============================================================================
# 🏁 7. 전체 데이터로 최종 모델 재학습 (피처 중요도 분석 및 저장용)
# =============================================================================

final_scaler = StandardScaler().fit(X)
final_models = {}
for name, base_model in models.items():
    m = base_model.__class__(**base_model.get_params())
    if name == 'lr':
        m.fit(final_scaler.transform(X), y)
    else:
        m.fit(X, y)
    final_models[name] = m

print("✅ 전체 train 데이터로 최종 모델 재학습 완료")

In [ ]:
# =============================================================================
# ⚖️ 8. 앙상블 가중치 최적화 (OOF 기준 연속 최적화)
#
#   기존 코드는 {0.1, 0.2, ..., 0.4} 6개 값의 3중 for문 그리드 서치였다.
#   AUC는 순위 기반의 계단 함수라 기울기가 대부분의 지점에서 0이므로,
#   SLSQP 같은 그래디언트 기반 방법은 초기값(균등 가중치)에서 전혀 움직이지
#   않고 그대로 반환해버린다(직접 검증해본 결과 실제로 그랬다).
#   대신 미분 없이 동작하는 differential_evolution을 쓰면 이 비매끄러운
#   목적함수에서도 실제로 더 나은 가중치를 찾아낸다.
# =============================================================================

oof_matrix = np.column_stack([oof_preds[n] for n in model_names])
n_models = len(model_names)

def neg_auc(weights):
    w = np.clip(weights, 0, None)
    total = w.sum()
    w = w / total if total > 0 else np.ones(n_models) / n_models
    return -roc_auc_score(y, oof_matrix @ w)

result = differential_evolution(
    neg_auc, bounds=[(0, 1)] * n_models,
    seed=RANDOM_STATE, maxiter=300, tol=1e-8, polish=True,
)
optimal_weights = np.clip(result.x, 0, None)
optimal_weights = optimal_weights / optimal_weights.sum()
ensemble_oof_auc = -neg_auc(optimal_weights)

print("   최적 가중치 (OOF 기준):")
for name, w in zip(model_names, optimal_weights):
    print(f"     {name.upper()}: {w:.3f}")
print(f"   가중 블렌드 OOF AUC: {ensemble_oof_auc:.4f}")

# 폴드별 앙상블 AUC (아래 통계 검정에서 개별 모델과 비교하기 위함)
ensemble_fold_aucs = np.array([
    roc_auc_score(y.iloc[val_idx], oof_matrix[val_idx] @ optimal_weights)
    for _, val_idx in fold_indices
])
print(f"   폴드별 앙상블 AUC: {[f'{a:.4f}' for a in ensemble_fold_aucs]}")

In [ ]:
# =============================================================================
# 🧠 9. 스태킹(메타모델)과 가중 블렌드 비교
#
#   OOF 예측을 입력으로 하는 로지스틱 회귀 메타모델(스태킹)을 가중 블렌드와
#   비교해서 더 나은 쪽을 최종 앙상블로 채택한다. 스태킹 성능은 OOF 위에서
#   다시 5-fold CV로 평가하므로(cross_val_score) 같은 데이터를 학습/평가에
#   이중으로 쓰는 것을 피한다.
# =============================================================================

meta_model = LogisticRegression(max_iter=1000)
stacking_cv_scores = cross_val_score(meta_model, oof_matrix, y, cv=cv, scoring='roc_auc')

print(f"   스태킹 메타모델 CV AUC: {stacking_cv_scores.mean():.4f} ± {stacking_cv_scores.std():.4f}")
print(f"   가중 블렌드 OOF AUC:     {ensemble_oof_auc:.4f}")

use_stacking = stacking_cv_scores.mean() > ensemble_oof_auc
if use_stacking:
    print("   ✅ 스태킹이 더 우수하여 최종 앙상블로 채택합니다.")
    meta_model.fit(oof_matrix, y)
    final_ensemble_auc = stacking_cv_scores.mean()
else:
    print("   ✅ 가중 블렌드가 더 우수하여(또는 동등하여) 최종 앙상블로 채택합니다.")
    final_ensemble_auc = ensemble_oof_auc

In [ ]:
# =============================================================================
# 📈 10. 피처 중요도 분석
# =============================================================================

rf_model = final_models['rf']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("   상위 15개 중요 피처:")
for i, (_, row) in enumerate(feature_importance.head(15).iterrows()):
    print(f"     {i+1:2d}. {row['feature']:<30}: {row['importance']:.4f}")

plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.title('상위 20개 피처 중요도 (Random Forest)')
plt.xlabel('중요도')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 🔬 11. 통계적 유의성 검증 (앙상블 vs 개별 모델)
#
#   fold_aucs / ensemble_fold_aucs는 모두 같은 5-fold 분할(cv)에서 나온
#   대응(paired) 표본이므로 대응표본 t-검정을 그대로 적용할 수 있다.
#   (기존 코드는 이 검정을 위해 cross_val_score로 모델을 다시 한 번씩
#    학습했는데, 6번 섹션에서 이미 만든 fold_aucs를 재사용하면 동일한 정보를
#    중복 학습 없이 얻을 수 있다.)
# =============================================================================

def ensemble_significance_test(individual_scores, ensemble_scores, alpha=0.05):
    print(f"🔬 앙상블 vs 개별 모델 통계적 검증 (α = {alpha})")
    print("=" * 60)
    results = {}
    for model_name, scores in individual_scores.items():
        t_stat, p_value = stats.ttest_rel(ensemble_scores, scores)
        pooled_std = np.sqrt((np.var(ensemble_scores) + np.var(scores)) / 2)
        cohens_d = (np.mean(ensemble_scores) - np.mean(scores)) / pooled_std if pooled_std > 0 else 0.0

        results[model_name] = {
            'ensemble_mean': np.mean(ensemble_scores),
            'individual_mean': np.mean(scores),
            'difference': np.mean(ensemble_scores) - np.mean(scores),
            't_statistic': t_stat,
            'p_value': p_value,
            'cohens_d': cohens_d,
            'significant': p_value < alpha,
        }

        print(f"{model_name.upper()}:")
        print(f"   앙상블 평균: {np.mean(ensemble_scores):.4f}")
        print(f"   개별 평균: {np.mean(scores):.4f}")
        print(f"   차이: {np.mean(ensemble_scores) - np.mean(scores):+.4f}")
        print(f"   t-통계량: {t_stat:.3f}")
        print(f"   p-value: {p_value:.4f}")
        print(f"   Cohen's d: {cohens_d:.3f}")
        print(f"   {'✅ 통계적으로 유의한 개선' if p_value < alpha else '❌ 통계적으로 유의하지 않음'} (α = {alpha})")
        print()
    return results

print("📊 모델별 5-fold 교차검증 성능 요약")
for name in model_names:
    scores = fold_aucs[name]
    mean_score, std_score = scores.mean(), scores.std()
    ci = 1.96 * std_score / np.sqrt(N_FOLDS)
    print(f"   {name.upper()}: {mean_score:.4f} ± {std_score:.4f}  (95% CI: [{mean_score - ci:.4f}, {mean_score + ci:.4f}])")

ens_mean, ens_std = ensemble_fold_aucs.mean(), ensemble_fold_aucs.std()
ens_ci = 1.96 * ens_std / np.sqrt(N_FOLDS)
print(f"   ENSEMBLE: {ens_mean:.4f} ± {ens_std:.4f}  (95% CI: [{ens_mean - ens_ci:.4f}, {ens_mean + ens_ci:.4f}])")

print()
significance_results = ensemble_significance_test(fold_aucs, ensemble_fold_aucs)

In [ ]:
# =============================================================================
# 📈 12. 통계적 분석 결과 시각화
# =============================================================================

def plot_statistical_results(ensemble_scores, individual_scores):
    plt.figure(figsize=(15, 8))

    plt.subplot(2, 3, 1)
    all_scores = list(individual_scores.values()) + [ensemble_scores]
    labels = [n.upper() for n in individual_scores.keys()] + ['ENSEMBLE']
    box_plot = plt.boxplot(all_scores, labels=labels, patch_artist=True)
    box_plot['boxes'][-1].set_facecolor('red')
    box_plot['boxes'][-1].set_alpha(0.7)
    plt.title('교차 검증 성능 분포')
    plt.ylabel('AUC Score')
    plt.xticks(rotation=45)

    plt.subplot(2, 3, 2)
    improvements, model_labels = [], []
    for name, scores in individual_scores.items():
        diff = ensemble_scores - scores
        improvements.extend(diff)
        model_labels.extend([name.upper()] * len(diff))
    df_improvements = pd.DataFrame({'Improvement': improvements, 'Model': model_labels})
    sns.boxplot(data=df_improvements, x='Model', y='Improvement')
    plt.title('앙상블 vs 개별 모델 성능 향상')
    plt.ylabel('AUC Improvement')
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)

    plt.subplot(2, 3, 3)
    means = [np.mean(s) for s in individual_scores.values()] + [np.mean(ensemble_scores)]
    errors = [np.std(s) for s in individual_scores.values()] + [np.std(ensemble_scores)]
    bars = plt.bar(range(len(labels)), means, yerr=errors, capsize=5, alpha=0.7)
    bars[-1].set_color('red')
    plt.xticks(range(len(labels)), labels, rotation=45)
    plt.title('평균 성능 비교 (± 표준편차)')
    plt.ylabel('AUC Score')

    plt.subplot(2, 3, 4)
    all_scores_dict = dict(individual_scores)
    all_scores_dict['ensemble'] = ensemble_scores
    model_list = list(all_scores_dict.keys())
    p_values = np.ones((len(model_list), len(model_list)))
    for i, m1 in enumerate(model_list):
        for j, m2 in enumerate(model_list):
            if i != j:
                _, p_val = stats.ttest_rel(all_scores_dict[m1], all_scores_dict[m2])
                p_values[i, j] = p_val
    sns.heatmap(p_values, annot=True, fmt='.3f', cmap='RdYlBu_r',
                xticklabels=[m.upper() for m in model_list],
                yticklabels=[m.upper() for m in model_list])
    plt.title('대응표본 t-검정 p-values')

    plt.tight_layout()
    plt.show()

plot_statistical_results(ensemble_fold_aucs, fold_aucs)

In [ ]:
# =============================================================================
# 🔮 13. 테스트 예측 및 제출 파일 생성
# =============================================================================

test_matrix = np.column_stack([test_preds[n] for n in model_names])

if use_stacking:
    final_test_pred = meta_model.predict_proba(test_matrix)[:, 1]
else:
    final_test_pred = test_matrix @ optimal_weights

submission = pd.DataFrame({'id': test['id'], 'Exited': final_test_pred})
submission.to_csv('advanced_ensemble_submission.csv', index=False)
print("✅ 제출 파일 저장 완료: advanced_ensemble_submission.csv")
print(f"   예측 이탈률: {final_test_pred.mean():.2%}  (실제 train 이탈률: {train['Exited'].mean():.2%})")

if _in_colab():
    from google.colab import files
    files.download('advanced_ensemble_submission.csv')

In [ ]:
# =============================================================================
# 💾 14. 모델 아티팩트 저장 (재사용/배포용)
# =============================================================================

artifact = {
    'final_models': final_models,
    'final_scaler': final_scaler,
    'model_names': model_names,
    'optimal_weights': optimal_weights,
    'meta_model': meta_model if use_stacking else None,
    'use_stacking': use_stacking,
    'balance_q80': balance_q80,
    'feature_cols': feature_cols,
}
joblib.dump(artifact, 'churn_ensemble_artifact.joblib')
print("✅ 모델 아티팩트 저장 완료: churn_ensemble_artifact.joblib")

In [ ]:
# =============================================================================
# 🎉 15. 최종 결과 요약
# =============================================================================

print("=" * 60)
print("🎉 고급 앙상블 모델 완료!")
print("=" * 60)
print(f"최종 앙상블 방식: {'스태킹(메타모델)' if use_stacking else '가중 블렌드'}")
print(f"최종 앙상블 CV AUC: {final_ensemble_auc:.4f}")
print(f"예측 이탈률: {final_test_pred.mean():.2%}")
print(f"실제 이탈률: {train['Exited'].mean():.2%}")

print("\n📊 개별 모델 5-fold 평균 AUC:")
for name in model_names:
    print(f"   {name.upper()}: {fold_aucs[name].mean():.4f} ± {fold_aucs[name].std():.4f}")

print("\n🎯 핵심 인사이트:")
top_3 = feature_importance.head(3)['feature'].tolist()
print(f"   가장 중요한 피처: {', '.join(top_3)}")

significant_improvements = sum(1 for r in significance_results.values() if r['significant'] and r['difference'] > 0)
print(f"\n📈 통계적 유의성: 개별 모델 {len(model_names)}개 중 {significant_improvements}개 대비 앙상블이 유의하게 우수 (p < 0.05)")